In [0]:
%run ./log

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
import time
logger=get_logger("AutoloaderLogger")

class AutoloaderIngestion:

    def __init__(self,spark,source_container,target_container,checkpoint_container,filename):
        self.spark=spark
        self.source_container=source_container
        self.target_container=target_container
        self.checkpoint_container=checkpoint_container
        self.file=filename
        

    def get_path(self):
        src=f"abfss://{self.source_container}@bhupeshstorage001.dfs.core.windows.net/{self.file}/"
        tgt=f"abfss://{self.target_container}@bhupeshstorage001.dfs.core.windows.net/{self.file}/"  
        chk=f"abfss://{self.checkpoint_container}@bhupeshstorage001.dfs.core.windows.net/checkpoint/{self.file}/" 
        schema_loc=f"abfss://{self.checkpoint_container}@bhupeshstorage001.dfs.core.windows.net/schema/{self.file}/" 

        logger.info(f"[PATH] Source = {src}")
        logger.info(f"[PATH] Target = {tgt}")
        logger.info(f"[PATH] Checkpoint = {chk}")
         
   

        return  src, tgt, chk, schema_loc 
    


    def is_empty(self,path):
        try:

            files=dbutils.fs.ls(path)
            if len(files)==0:
                logger.warning(f" Folder is empty: {path}")
                return True
            return False
            
        except Exception as e:
            logger.warning(f" Cannot access {path}: {e}")
            return True
            
    def read_stream(self,src, schema_loc, retries=3, wait=4):

        schema = StructType([
            StructField("snapshotIndex", IntegerType(), True),
            StructField("userID", StringType(), True),
            StructField("bitrateWatchedKbps", IntegerType(), True),
            StructField("startBuffering", StringType(), True),
            StructField("endBuffering", StringType(), True)
             ])
        
        if self.is_empty(src):
            logger.warning("Skipping read  directory  is empty.")
            return None
        
        attempt=1

        while attempt<=retries:
            try:
                logger.info(f"Reading stream from {src}")


                df=(self.spark.readStream
                    .format("cloudFiles")
                    .option("cloudFiles.format","json")
                    .option("cloudFiles.schemaLocation", schema_loc)
                    .option("cloudFiles.schemaLocation",schema_loc)
                    .option("cloudFiles.includeExistingFiles", "true")
                    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
                    .option("rescuedDataColumn", "_rescued_data")
                    .option("cloudFiles.validateOptions", "true")
                    .option("multiLine", "true")
                    .load(src)
                    
                    
                    )
                
                df=(df.withColumn("ingesttime",current_timestamp())
                        .withColumn("filename",col("_metadata.file_path"))
                        .withColumn("filedate",current_date()))


                logger.info("Auto Loader stream initialized") 
                return df   
                

            except Exception as e:
                logger.warning(f"Error reading stream wait for retry: {e}") 


            if attempt<retries:
                time.sleep(wait)
                attempt+=1

            else:
                logger.error(f"Error reading stream after {retries} retries: {e}")
                return None    


    def write_stream(self,df,tgt,chk):

        if df is None:
          logger.warning(f"Skipping '{self.file}' — read_stream failed.")
          return

        try:
            logger.info(f" Writing Bronze data  {tgt}")

            query=(df.writeStream
                    .format("delta")
                    .option("checkpointLocation",chk)
                    .outputMode("append")
                    .trigger(availableNow=True)
                    .start(tgt))


            query.awaitTermination()
            self.spark.sql(f"""
                       CREATE TABLE IF NOT EXISTS hive_streamming.bronze.{self.file}
                       USING DELTA
                       LOCATION '{tgt}'
                       """)   
            
            
            logger.info(f" Writing Bronze data  {tgt} completed")

        except Exception as e:
            logger.error(f"Error writing stream: {e}")
            return None 


def main():
        logger.info(" Starting Auto Loader Bronze Pipeline")

        dbutils.widgets.text("file_name","")
        dataset=dbutils.widgets.get("file_name")

        if not dataset:
            logger.error("No dataset name are provided")
            return

        ingestor= AutoloaderIngestion(
            spark=spark,
            source_container="raw",
            target_container="bronze",
            checkpoint_container="checkpoint",
            filename=dataset
        )   

        src, tgt, chk, schema_loc = ingestor.get_path()
        df=ingestor.read_stream(src, schema_loc)
        ingestor.write_stream(df,tgt,chk)
        logger.info(" Auto Loader Bronze Pipeline completed")


if __name__ == "__main__":
    main()    
     




              





       


             




